# 09 — Recommendations


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from build_optimiser.config import Config
from build_optimiser.graph import load_graph, attach_metrics
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

cfg = Config.from_yaml('../config.yaml')
file_df = pd.read_parquet('../data/processed/file_metrics.parquet')
target_df = pd.read_parquet('../data/processed/target_metrics.parquet')
edge_df = pd.read_parquet('../data/processed/edge_list.parquet')
G = load_graph('../data/processed/edge_list.parquet')
attach_metrics(G, target_df)

%matplotlib inline
sns.set_theme(style='whitegrid')

## Compile Candidate List

Merge, split, individual target optimisations, codegen optimisations.


In [ ]:
import json
import os
from collections import defaultdict

import igraph as ig
import leidenalg
import numpy as np
from networkx.algorithms.community import louvain_communities, modularity

from build_optimiser.simulation import (
    codegen_cascade_cost,
    rebuild_cost,
    simulate_merge,
)

# ── Effort estimates (engineer-days) ─────────────────────────────────────────
EFFORT_SPLIT = 2.0
EFFORT_MERGE = 1.0
EFFORT_UNUSED = 0.5
EFFORT_IWYU = 3.0
EFFORT_PCH = 1.5
EFFORT_CODEGEN = 2.0

# ═══════════════════════════════════════════════════════════════════════════════
# Re-derive critical path (NB03 logic)
# ═══════════════════════════════════════════════════════════════════════════════
weight_attr = 'effective_weight'
for n in G.nodes():
    nd = G.nodes[n]
    G.nodes[n][weight_attr] = (
        (nd.get('codegen_time_ms') or 0)
        + (nd.get('compile_time_max_ms') or 0)
        + (nd.get('archive_time_ms') or 0)
        + (nd.get('link_time_ms') or 0)
    )

topo = list(nx.topological_sort(G))
earliest_finish = {}
predecessor_on_path = {}

for n in reversed(topo):  # build order: dependencies first
    w = G.nodes[n][weight_attr]
    deps = list(G.successors(n))
    if not deps:
        earliest_finish[n] = w
        predecessor_on_path[n] = None
    else:
        best_dep = max(deps, key=lambda d: earliest_finish[d])
        earliest_finish[n] = earliest_finish[best_dep] + w
        predecessor_on_path[n] = best_dep

cp_end = max(earliest_finish, key=earliest_finish.get)
cp_length = earliest_finish[cp_end]

path = []
node = cp_end
while node is not None:
    path.append(node)
    node = predecessor_on_path[node]
cp_nodes = path
cp_set = set(cp_nodes)

# Backward pass — slack
earliest_start = {}
for n in reversed(topo):
    deps = list(G.successors(n))
    earliest_start[n] = max((earliest_finish[d] for d in deps), default=0)

latest_finish = {}
latest_start = {}
for n in topo:
    w = G.nodes[n][weight_attr]
    dependants = list(G.predecessors(n))
    if not dependants:
        latest_finish[n] = cp_length
    else:
        latest_finish[n] = min(latest_start[d] for d in dependants)
    latest_start[n] = latest_finish[n] - w

slack = {n: latest_start[n] - earliest_start[n] for n in G.nodes()}

print(f"Critical path: {' → '.join(cp_nodes)}  ({cp_length:,} ms)")

# ═══════════════════════════════════════════════════════════════════════════════
# Re-derive cost_df (NB05 logic)
# ═══════════════════════════════════════════════════════════════════════════════
working_days = cfg.git_history_months * 20

cost_rows = []
for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    commit_count = row.get('git_commit_count_total', 0) or 0
    change_prob = min(commit_count / working_days, 1.0) if working_days > 0 else 0.0
    r_cost = rebuild_cost(G, t, target_df)
    expected = change_prob * r_cost
    cost_rows.append({
        'cmake_target': t,
        'change_prob': change_prob,
        'rebuild_cost_ms': r_cost,
        'expected_daily_cost_ms': expected,
        'has_codegen': (row.get('codegen_file_count') or 0) > 0,
    })

cost_df = pd.DataFrame(cost_rows).sort_values('expected_daily_cost_ms', ascending=False)
cost_lookup = cost_df.set_index('cmake_target')

total_expected = cost_df['expected_daily_cost_ms'].sum()
print(f"Expected daily rebuild cost (current): {total_expected:,.0f} ms ({total_expected / 1000:.1f} s)")

# ═══════════════════════════════════════════════════════════════════════════════
# Candidate Source 1 — Split candidates (from NB07 persisted results)
# ═══════════════════════════════════════════════════════════════════════════════
split_recs = pd.read_parquet('../data/results/split_recommendations.parquet')
actionable_splits = split_recs[split_recs['recommended_k'] > 1].copy()

candidates = []
for _, row in actionable_splits.iterrows():
    t = row['cmake_target']
    candidates.append({
        'type': 'split',
        'cmake_target': t,
        'saving_ms': int(row['compile_saving_ms']),
        'effort_days': EFFORT_SPLIT,
        'on_cp': t in cp_set,
        'detail': (f"Split into {int(row['recommended_k'])} partitions "
                   f"({row['method']}, balance={row['balance_ratio']:.2f}, "
                   f"cross_includes={int(row['cross_partition_includes'])})"),
    })

print(f"\nSplit candidates: {len(actionable_splits)}")

# ═══════════════════════════════════════════════════════════════════════════════
# Candidate Source 2 — Merge candidates (re-derive NB04 community detection)
# ═══════════════════════════════════════════════════════════════════════════════
direct_edges = edge_df[edge_df['is_direct']]
G_und = nx.Graph()
G_und.add_nodes_from(G.nodes())
G_und.add_edges_from(zip(direct_edges['source_target'], direct_edges['dest_target']))

# Louvain
louvain_comms = louvain_communities(G_und, resolution=1.0, seed=42)
louvain_mod = modularity(G_und, louvain_comms)

# Leiden
ig_g = ig.Graph.from_networkx(G_und)
_node_names = ig_g.vs['_nx_name']
leiden_part = leidenalg.find_partition(ig_g, leidenalg.ModularityVertexPartition, seed=42)
leiden_node_comm = {_node_names[i]: leiden_part.membership[i] for i in range(ig_g.vcount())}
leiden_comms = [
    {n for n, c in leiden_node_comm.items() if c == cid}
    for cid in range(max(leiden_part.membership) + 1)
]
leiden_mod = modularity(G_und, leiden_comms)

if leiden_mod >= louvain_mod:
    communities = leiden_comms
else:
    communities = louvain_comms

# Community characterisation — cohesion / coupling
target_metrics_idx = target_df.set_index('cmake_target')
comm_rows = []
for cid, comm in enumerate(communities):
    subg = G_und.subgraph(comm)
    internal_edges = subg.number_of_edges()
    external_edges = sum(1 for u in comm for v in G_und.neighbors(u) if v not in comm)
    total_edges = internal_edges + external_edges
    cohesion = internal_edges / total_edges if total_edges > 0 else 0.0
    coupling = external_edges / total_edges if total_edges > 0 else 0.0
    comm_rows.append({
        'community_id': cid,
        'targets': sorted(comm),
        'target_count': len(comm),
        'cohesion_ratio': cohesion,
        'coupling_ratio': coupling,
    })

comm_df = pd.DataFrame(comm_rows)

COHESION_THRESHOLD = comm_df['cohesion_ratio'].quantile(0.50)
COUPLING_THRESHOLD = comm_df['coupling_ratio'].quantile(0.50)

merge_candidates = comm_df[
    (comm_df['cohesion_ratio'] >= COHESION_THRESHOLD)
    & (comm_df['coupling_ratio'] <= COUPLING_THRESHOLD)
    & (comm_df['target_count'] >= 2)
]

for _, row in merge_candidates.iterrows():
    targets = [t for t in row['targets'] if t in target_metrics_idx.index]
    if len(targets) < 2:
        continue
    result = simulate_merge(G, targets, target_df)
    candidates.append({
        'type': 'merge',
        'cmake_target': ', '.join(sorted(targets)),
        'saving_ms': result['savings_ms'],
        'effort_days': EFFORT_MERGE,
        'on_cp': any(t in cp_set for t in targets),
        'detail': (f"Merge {len(targets)} targets "
                   f"(cohesion={row['cohesion_ratio']:.2f}, "
                   f"coupling={row['coupling_ratio']:.2f}). "
                   f"{'; '.join(result['notes'])}" if result['notes'] else ''),
    })

print(f"Merge candidates: {len(merge_candidates)}")

# ═══════════════════════════════════════════════════════════════════════════════
# Candidate Source 3 — Unused dependencies
# ═══════════════════════════════════════════════════════════════════════════════
# For each direct edge, check if any source file in source_target includes a
# header that belongs to dest_target's source directories.

# Build target -> set of source directory prefixes
target_dirs = defaultdict(set)
for _, row in file_df.iterrows():
    d = os.path.normpath(os.path.dirname(row['source_file']))
    target_dirs[row['cmake_target']].add(d)

# Build target -> set of normalised included paths (from header_tree JSON)
target_includes = defaultdict(set)
for _, row in file_df.iterrows():
    ht = row.get('header_tree')
    if pd.isna(ht) if isinstance(ht, float) else not ht:
        continue
    try:
        tree = json.loads(ht) if isinstance(ht, str) else ht
    except (json.JSONDecodeError, TypeError):
        continue
    for entry in tree:
        if isinstance(entry, list) and len(entry) >= 2:
            target_includes[row['cmake_target']].add(os.path.normpath(str(entry[1])))

direct_link_edges = edge_df[(edge_df['is_direct']) & (edge_df['dependency_type'] == 'link')]
unused_dep_count = 0
for _, edge_row in direct_link_edges.iterrows():
    src_t = edge_row['source_target']
    dst_t = edge_row['dest_target']
    dst_dirs = target_dirs.get(dst_t, set())
    if not dst_dirs:
        continue  # e.g. INTERFACE library with no source files
    src_includes = target_includes.get(src_t, set())
    has_include_backing = any(
        any(inc.startswith(d) for d in dst_dirs)
        for inc in src_includes
    )
    if not has_include_backing:
        candidates.append({
            'type': 'unused_dep',
            'cmake_target': f"{src_t} → {dst_t}",
            'saving_ms': 0,
            'effort_days': EFFORT_UNUSED,
            'on_cp': src_t in cp_set or dst_t in cp_set,
            'detail': f"No #include backing found for link dependency {src_t} → {dst_t}",
        })
        unused_dep_count += 1

print(f"Unused dependency candidates: {unused_dep_count}")

# ═══════════════════════════════════════════════════════════════════════════════
# Candidate Source 4 — Codegen optimisations
# ═══════════════════════════════════════════════════════════════════════════════
codegen_targets = target_df[target_df['codegen_file_count'].fillna(0) > 0].copy()

for _, row in codegen_targets.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    cascade = codegen_cascade_cost(G, t, target_df)
    # Caching candidate
    if cascade > 0:
        candidates.append({
            'type': 'codegen_caching',
            'cmake_target': t,
            'saving_ms': int(row.get('codegen_time_ms') or 0),
            'effort_days': EFFORT_CODEGEN,
            'on_cp': t in cp_set,
            'detail': f"Cache codegen output (cascade cost={cascade:,} ms)",
        })
    # Isolation candidate — wrap in interface library if many dependants
    dependant_count = int(row.get('direct_dependant_count') or 0) + int(row.get('transitive_dependant_count') or 0)
    if dependant_count >= 3:
        candidates.append({
            'type': 'codegen_isolation',
            'cmake_target': t,
            'saving_ms': 0,
            'effort_days': EFFORT_CODEGEN,
            'on_cp': t in cp_set,
            'detail': f"Isolate codegen behind interface library ({dependant_count} dependants)",
        })

print(f"Codegen candidates: {sum(1 for c in candidates if c['type'].startswith('codegen_'))}")

# ═══════════════════════════════════════════════════════════════════════════════
# Candidate Source 5 — IWYU and PCH individual target optimisations
# ═══════════════════════════════════════════════════════════════════════════════
IWYU_EXPANSION_THRESHOLD = target_df['expansion_ratio_mean'].quantile(0.75)
PCH_DEPTH_THRESHOLD = 18

for _, row in target_df.iterrows():
    t = row['cmake_target']
    if t not in G:
        continue
    er = row.get('expansion_ratio_mean') or 0
    hd = row.get('header_depth_max') or 0

    if t in cp_set and er > IWYU_EXPANSION_THRESHOLD and er > 0:
        compile_ms = int(row.get('compile_time_sum_ms') or 0)
        candidates.append({
            'type': 'iwyu',
            'cmake_target': t,
            'saving_ms': int(compile_ms * 0.20),  # estimated 20% reduction
            'effort_days': EFFORT_IWYU,
            'on_cp': True,
            'detail': (f"IWYU cleanup: expansion_ratio={er:.0f}x "
                       f"(threshold={IWYU_EXPANSION_THRESHOLD:.0f}x), "
                       f"compile={compile_ms:,} ms"),
        })

    if hd >= PCH_DEPTH_THRESHOLD and (row.get('compile_time_max_ms') or 0) > 0:
        compile_ms = int(row.get('compile_time_sum_ms') or 0)
        candidates.append({
            'type': 'pch',
            'cmake_target': t,
            'saving_ms': int(compile_ms * 0.15),  # estimated 15% reduction
            'effort_days': EFFORT_PCH,
            'on_cp': t in cp_set,
            'detail': (f"Precompiled header: header_depth={int(hd)} "
                       f"(threshold={PCH_DEPTH_THRESHOLD}), "
                       f"compile={compile_ms:,} ms"),
        })

print(f"IWYU candidates: {sum(1 for c in candidates if c['type'] == 'iwyu')}")
print(f"PCH candidates: {sum(1 for c in candidates if c['type'] == 'pch')}")

# ═══════════════════════════════════════════════════════════════════════════════
# Assemble candidate_df
# ═══════════════════════════════════════════════════════════════════════════════
candidate_df = pd.DataFrame(candidates)
candidate_df.insert(0, 'id', range(1, len(candidate_df) + 1))

print(f"\n{'=' * 60}")
print(f"Total candidates: {len(candidate_df)}")
print(f"{'=' * 60}")
print(candidate_df.groupby('type')['id'].count().rename('count').to_string())
print()
display(candidate_df)

## Score Each Candidate

Expected build time improvement from change impact simulation.


In [ ]:
from matplotlib.patches import Patch

# ── Score each candidate by expected daily build time saving ─────────────────
TYPE_COLORS = {
    'split': 'steelblue',
    'merge': '#55A868',
    'unused_dep': '#C44E52',
    'codegen_caching': 'darkorange',
    'codegen_isolation': '#E8A84C',
    'iwyu': '#8172B2',
    'pch': '#CCB974',
}

scored_rows = []
for _, row in candidate_df.iterrows():
    ctype = row['type']
    target_str = row['cmake_target']
    saving_ms = row['saving_ms']

    # Look up change probability — for multi-target candidates, use mean
    targets_in_candidate = [t.strip() for t in target_str.split(',')]
    probs = [cost_lookup.loc[t, 'change_prob']
             for t in targets_in_candidate if t in cost_lookup.index]
    mean_prob = np.mean(probs) if probs else 0.0

    if ctype == 'split':
        # High confidence: measured compile saving * change probability
        daily = mean_prob * saving_ms
        confidence = 'High'

    elif ctype == 'merge':
        # Medium confidence: conservative archive/link savings * mean change prob
        daily = mean_prob * saving_ms
        confidence = 'Medium'

    elif ctype == 'unused_dep':
        # Maintenance benefit, not direct time saving
        daily = 0.0
        confidence = 'Low'

    elif ctype == 'codegen_caching':
        # Low confidence: codegen time saved when codegen would run
        daily = mean_prob * saving_ms
        confidence = 'Low'

    elif ctype == 'codegen_isolation':
        # Structural improvement — no direct time saving
        daily = 0.0
        confidence = 'Low'

    elif ctype == 'iwyu':
        # Low confidence: estimated 20% compile reduction * change prob
        daily = mean_prob * saving_ms
        confidence = 'Low'

    elif ctype == 'pch':
        # Low confidence: estimated 15% compile reduction * change prob
        daily = mean_prob * saving_ms
        confidence = 'Low'

    else:
        daily = 0.0
        confidence = 'Low'

    scored_rows.append({
        **row.to_dict(),
        'time_saving_ms_per_day': round(daily, 2),
        'total_saving_ms': saving_ms,
        'confidence': confidence,
    })

scored_df = pd.DataFrame(scored_rows).sort_values('time_saving_ms_per_day', ascending=False)

print("=== Scored Candidates (top 15) ===")
display(scored_df[['id', 'type', 'cmake_target', 'time_saving_ms_per_day',
                    'total_saving_ms', 'effort_days', 'confidence']].head(15).reset_index(drop=True))

# ── Two-panel plot ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Panel 1: Top 15 by daily saving
top15 = scored_df[scored_df['time_saving_ms_per_day'] > 0].head(15)
if not top15.empty:
    colors = [TYPE_COLORS.get(t, 'grey') for t in top15['type']]
    ax1.barh(range(len(top15)), top15['time_saving_ms_per_day'].values, color=colors)
    ax1.set_yticks(range(len(top15)))
    ax1.set_yticklabels(
        [f"[{r['type']}] {r['cmake_target'][:30]}" for _, r in top15.iterrows()],
        fontsize=8,
    )
    ax1.invert_yaxis()
    ax1.set_xlabel('Expected daily saving (ms/day)')
    ax1.set_title('Top 15 Candidates by Daily Build Time Saving')
    legend_types = top15['type'].unique()
    ax1.legend(
        handles=[Patch(color=TYPE_COLORS.get(t, 'grey'), label=t) for t in legend_types],
        loc='lower right', fontsize=8,
    )
else:
    ax1.text(0.5, 0.5, 'No candidates with positive daily saving',
             transform=ax1.transAxes, ha='center')

# Panel 2: Total saving vs effort, sized by daily saving
has_saving = scored_df[scored_df['total_saving_ms'] > 0].copy()
if not has_saving.empty:
    sizes = (has_saving['time_saving_ms_per_day'] / has_saving['time_saving_ms_per_day'].max() * 300 + 30)
    colors2 = [TYPE_COLORS.get(t, 'grey') for t in has_saving['type']]
    ax2.scatter(has_saving['effort_days'], has_saving['total_saving_ms'],
                s=sizes.values, c=colors2, alpha=0.7, edgecolors='black', linewidths=0.5)
    for _, r in has_saving.iterrows():
        ax2.annotate(r['cmake_target'][:20], (r['effort_days'], r['total_saving_ms']),
                     fontsize=7, ha='left', va='bottom')
    ax2.set_xlabel('Effort (engineer-days)')
    ax2.set_ylabel('Total saving (ms)')
    ax2.set_title('Total Saving vs Effort (size = daily saving)')
else:
    ax2.text(0.5, 0.5, 'No candidates with measurable savings',
             transform=ax2.transAxes, ha='center')

plt.tight_layout()
plt.show()

# Summary stats
positive = scored_df[scored_df['time_saving_ms_per_day'] > 0]
print(f"\nCandidates with positive daily saving: {len(positive)} / {len(scored_df)}")
print(f"Total potential daily saving: {positive['time_saving_ms_per_day'].sum():,.1f} ms/day")
print(f"Total effort required: {positive['effort_days'].sum():.1f} engineer-days")

## Rank by ROI

(expected build time saved per day) / (estimated implementation effort).


In [ ]:
# ── ROI = daily saving / effort ──────────────────────────────────────────────
roi_df = scored_df.copy()
roi_df['roi'] = roi_df['time_saving_ms_per_day'] / roi_df['effort_days'].clip(lower=0.1)
roi_df = roi_df.sort_values('roi', ascending=False).reset_index(drop=True)
roi_df['rank'] = range(1, len(roi_df) + 1)

# Tier assignment
roi_median = roi_df.loc[roi_df['roi'] > 0, 'roi'].median() if (roi_df['roi'] > 0).any() else 0

def assign_tier(row):
    if row['roi'] > roi_median and row['on_cp']:
        return 'Tier 1 — Priority'
    elif row['roi'] > roi_median:
        return 'Tier 2 — High ROI'
    else:
        return 'Tier 3 — Other'

roi_df['tier'] = roi_df.apply(assign_tier, axis=1)
roi_df['priority'] = roi_df['tier'] == 'Tier 1 — Priority'

TIER_COLORS = {
    'Tier 1 — Priority': 'crimson',
    'Tier 2 — High ROI': 'darkorange',
    'Tier 3 — Other': 'steelblue',
}

# ── Two-panel plot ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel 1: ROI bar chart — all candidates with ROI > 0
roi_positive = roi_df[roi_df['roi'] > 0].copy()
if not roi_positive.empty:
    colors = [TIER_COLORS[t] for t in roi_positive['tier']]
    ax1.barh(range(len(roi_positive)), roi_positive['roi'].values, color=colors)
    ax1.set_yticks(range(len(roi_positive)))
    ax1.set_yticklabels(
        [f"#{r['rank']} [{r['type']}] {r['cmake_target'][:25]}"
         for _, r in roi_positive.iterrows()],
        fontsize=8,
    )
    ax1.invert_yaxis()
    if roi_median > 0:
        ax1.axvline(roi_median, color='black', linestyle='--', alpha=0.6,
                     label=f'Median ROI ({roi_median:.2f})')
    ax1.set_xlabel('ROI (ms saved per day / effort day)')
    ax1.set_title('Candidates Ranked by ROI')
    ax1.legend(
        handles=[Patch(color=c, label=t) for t, c in TIER_COLORS.items()]
        + ([ax1.get_lines()[0]] if roi_median > 0 else []),
        loc='lower right', fontsize=8,
    )
else:
    ax1.text(0.5, 0.5, 'No candidates with positive ROI', transform=ax1.transAxes, ha='center')

# Panel 2: ROI vs daily saving scatter, marker by confidence
confidence_markers = {'High': 'o', 'Medium': 's', 'Low': 'D'}
for conf, marker in confidence_markers.items():
    sub = roi_positive[roi_positive['confidence'] == conf] if not roi_positive.empty else pd.DataFrame()
    if not sub.empty:
        colors_s = [TYPE_COLORS.get(t, 'grey') for t in sub['type']]
        ax2.scatter(sub['time_saving_ms_per_day'], sub['roi'],
                    s=80, c=colors_s, marker=marker, alpha=0.7,
                    edgecolors='black', linewidths=0.5, label=f'{conf} confidence')
        for _, r in sub.iterrows():
            ax2.annotate(r['cmake_target'][:18],
                         (r['time_saving_ms_per_day'], r['roi']),
                         fontsize=7, ha='left', va='bottom')

ax2.set_xlabel('Daily saving (ms/day)')
ax2.set_ylabel('ROI (ms/day per effort day)')
ax2.set_title('ROI vs Daily Saving (shape = confidence)')
if not roi_positive.empty:
    ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

# ── Tier breakdown ───────────────────────────────────────────────────────────
for tier_name in ['Tier 1 — Priority', 'Tier 2 — High ROI', 'Tier 3 — Other']:
    tier_rows = roi_df[roi_df['tier'] == tier_name]
    print(f"\n{'=' * 60}")
    print(f"  {tier_name}  ({len(tier_rows)} candidates)")
    print(f"{'=' * 60}")
    for _, r in tier_rows.iterrows():
        cp_flag = ' [CP]' if r['on_cp'] else ''
        print(f"  #{r['rank']:2d}  [{r['type']:18s}] {r['cmake_target'][:35]:35s}"
              f"  ROI={r['roi']:7.2f}  save={r['time_saving_ms_per_day']:7.1f} ms/day"
              f"  effort={r['effort_days']:.1f}d  {r['confidence']:6s}{cp_flag}")

## Quick Wins

Unused deps, high expansion ratio on critical path, high header depth, codegen on critical path.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 1 — Unused dependencies
# ═══════════════════════════════════════════════════════════════════════════════
qw_unused = candidate_df[candidate_df['type'] == 'unused_dep'].copy()
if not qw_unused.empty:
    # Annotate with source target's expected daily cost
    qw_unused['source_target'] = qw_unused['cmake_target'].str.split(' → ').str[0]
    qw_unused['source_daily_cost_ms'] = qw_unused['source_target'].map(
        cost_lookup['expected_daily_cost_ms']
    ).fillna(0)
    qw_unused = qw_unused.sort_values('source_daily_cost_ms', ascending=False)

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 2 — IWYU candidates (CP targets with high expansion ratio)
# ═══════════════════════════════════════════════════════════════════════════════
IWYU_EXPANSION_THRESHOLD_QW = target_df['expansion_ratio_mean'].quantile(0.75)
qw_iwyu = target_df[
    (target_df['cmake_target'].isin(cp_set))
    & (target_df['expansion_ratio_mean'] > IWYU_EXPANSION_THRESHOLD_QW)
    & (target_df['expansion_ratio_mean'] > 0)
][['cmake_target', 'expansion_ratio_mean', 'compile_time_sum_ms',
   'compile_time_max_ms', 'preprocessed_bytes_total']].copy()
qw_iwyu = qw_iwyu.sort_values('expansion_ratio_mean', ascending=False)

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 3 — PCH candidates (high header depth)
# ═══════════════════════════════════════════════════════════════════════════════
PCH_DEPTH_QW = 18
qw_pch = target_df[
    (target_df['header_depth_max'] >= PCH_DEPTH_QW)
    & (target_df['compile_time_max_ms'] > 0)
][['cmake_target', 'header_depth_max', 'unique_headers_total',
   'compile_time_sum_ms']].copy()
qw_pch['on_cp'] = qw_pch['cmake_target'].isin(cp_set)
qw_pch = qw_pch.sort_values(['on_cp', 'compile_time_sum_ms'], ascending=[False, False])

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 4 — Codegen on critical path
# ═══════════════════════════════════════════════════════════════════════════════
qw_codegen_cp = target_df[
    (target_df['codegen_time_ms'].fillna(0) > 0)
    & (target_df['cmake_target'].isin(cp_set))
][['cmake_target', 'codegen_time_ms', 'codegen_file_count',
   'compile_time_max_ms']].copy()

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 5 — Generated file bloat (high preprocessed size)
# ═══════════════════════════════════════════════════════════════════════════════
gen_files = file_df[file_df['is_generated'] & (file_df['compile_time_ms'] > 0)].copy()
if not gen_files.empty:
    pp_threshold = file_df.loc[file_df['compile_time_ms'] > 0, 'preprocessed_bytes'].quantile(0.75)
    qw_gen_bloat = gen_files[gen_files['preprocessed_bytes'] > pp_threshold][
        ['source_file', 'cmake_target', 'preprocessed_bytes', 'expansion_ratio',
         'compile_time_ms']
    ].sort_values('preprocessed_bytes', ascending=False)
else:
    qw_gen_bloat = pd.DataFrame()

# ═══════════════════════════════════════════════════════════════════════════════
# Quick Win 6 — High codegen ratio targets
# ═══════════════════════════════════════════════════════════════════════════════
CODEGEN_RATIO_THRESHOLD = 0.5
qw_high_codegen = target_df[
    (target_df['codegen_ratio'].fillna(0) >= CODEGEN_RATIO_THRESHOLD)
    & (target_df['codegen_compile_time_sum_ms'].fillna(0) > 0)
][['cmake_target', 'codegen_ratio', 'codegen_compile_time_sum_ms',
   'compile_time_sum_ms', 'codegen_file_count']].copy()
qw_high_codegen = qw_high_codegen.sort_values('codegen_compile_time_sum_ms', ascending=False)

# ═══════════════════════════════════════════════════════════════════════════════
# Consolidated quick wins summary
# ═══════════════════════════════════════════════════════════════════════════════
qw_rows = []
for _, r in qw_unused.iterrows():
    qw_rows.append({'category': 'Unused dependency', 'target': r['cmake_target'],
                     'description': r['detail'], 'effort_days': EFFORT_UNUSED})
for _, r in qw_iwyu.iterrows():
    qw_rows.append({'category': 'IWYU cleanup', 'target': r['cmake_target'],
                     'description': f"expansion_ratio={r['expansion_ratio_mean']:.0f}x, on CP",
                     'effort_days': EFFORT_IWYU})
for _, r in qw_pch.iterrows():
    qw_rows.append({'category': 'Precompiled header', 'target': r['cmake_target'],
                     'description': f"header_depth={int(r['header_depth_max'])}",
                     'effort_days': EFFORT_PCH})
for _, r in qw_codegen_cp.iterrows():
    qw_rows.append({'category': 'Codegen on CP', 'target': r['cmake_target'],
                     'description': f"codegen_time={int(r['codegen_time_ms'])} ms on critical path",
                     'effort_days': EFFORT_CODEGEN})
for _, r in qw_gen_bloat.iterrows():
    qw_rows.append({'category': 'Generated file bloat', 'target': r['cmake_target'],
                     'description': f"{r['source_file'].split('/')[-1]}: "
                                    f"{r['preprocessed_bytes'] / 1e6:.1f} MB preprocessed, "
                                    f"expansion={r['expansion_ratio']:.0f}x",
                     'effort_days': EFFORT_CODEGEN})
for _, r in qw_high_codegen.iterrows():
    qw_rows.append({'category': 'High codegen ratio', 'target': r['cmake_target'],
                     'description': f"codegen_ratio={r['codegen_ratio']:.0%}, "
                                    f"codegen_compile={int(r['codegen_compile_time_sum_ms'])} ms",
                     'effort_days': EFFORT_CODEGEN})

quick_wins_summary = pd.DataFrame(qw_rows) if qw_rows else pd.DataFrame(
    columns=['category', 'target', 'description', 'effort_days']
)

# ── Template recommendation guard ────────────────────────────────────────────
if target_df['gcc_template_time_sum_ms'].fillna(0).sum() == 0:
    print("Note: gcc_template_time_sum_ms is all-zero — template recommendations unavailable.\n")

# ── 2x3 plot grid ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Panel 1: Unused deps by source target daily cost
ax = axes[0, 0]
if not qw_unused.empty:
    ax.barh(range(len(qw_unused)), qw_unused['source_daily_cost_ms'].values, color='#C44E52')
    ax.set_yticks(range(len(qw_unused)))
    ax.set_yticklabels(qw_unused['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Source target daily cost (ms)')
else:
    ax.text(0.5, 0.5, 'None identified', transform=ax.transAxes, ha='center')
ax.set_title(f'Unused Dependencies ({len(qw_unused)})')

# Panel 2: IWYU — expansion ratio
ax = axes[0, 1]
if not qw_iwyu.empty:
    ax.barh(range(len(qw_iwyu)), qw_iwyu['expansion_ratio_mean'].values, color='#8172B2')
    ax.set_yticks(range(len(qw_iwyu)))
    ax.set_yticklabels(qw_iwyu['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Expansion ratio (mean)')
    ax.axvline(IWYU_EXPANSION_THRESHOLD_QW, color='red', linestyle='--', alpha=0.6,
               label=f'75th pctl ({IWYU_EXPANSION_THRESHOLD_QW:.0f}x)')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'None identified', transform=ax.transAxes, ha='center')
ax.set_title(f'IWYU Candidates on CP ({len(qw_iwyu)})')

# Panel 3: PCH — header depth
ax = axes[0, 2]
if not qw_pch.empty:
    colors_pch = ['crimson' if cp else 'steelblue' for cp in qw_pch['on_cp']]
    ax.barh(range(len(qw_pch)), qw_pch['header_depth_max'].values, color=colors_pch)
    ax.set_yticks(range(len(qw_pch)))
    ax.set_yticklabels(qw_pch['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Max header depth')
    ax.axvline(PCH_DEPTH_QW, color='red', linestyle='--', alpha=0.6, label=f'Threshold ({PCH_DEPTH_QW})')
    ax.legend(handles=[Patch(color='crimson', label='On CP'),
                       Patch(color='steelblue', label='Off CP')], fontsize=8)
else:
    ax.text(0.5, 0.5, 'None identified', transform=ax.transAxes, ha='center')
ax.set_title(f'PCH Candidates ({len(qw_pch)})')

# Panel 4: Codegen on CP
ax = axes[1, 0]
if not qw_codegen_cp.empty:
    ax.barh(range(len(qw_codegen_cp)), qw_codegen_cp['codegen_time_ms'].values, color='darkorange')
    ax.set_yticks(range(len(qw_codegen_cp)))
    ax.set_yticklabels(qw_codegen_cp['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Codegen time (ms)')
else:
    ax.text(0.5, 0.5, 'No codegen targets on CP', transform=ax.transAxes, ha='center')
ax.set_title(f'Codegen on Critical Path ({len(qw_codegen_cp)})')

# Panel 5: Generated file bloat
ax = axes[1, 1]
if not qw_gen_bloat.empty:
    labels = [f"{r['cmake_target']}/{r['source_file'].split('/')[-1]}"
              for _, r in qw_gen_bloat.iterrows()]
    ax.barh(range(len(qw_gen_bloat)), qw_gen_bloat['preprocessed_bytes'].values / 1e6,
            color='darkorange', alpha=0.8)
    ax.set_yticks(range(len(qw_gen_bloat)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlabel('Preprocessed size (MB)')
else:
    ax.text(0.5, 0.5, 'None identified', transform=ax.transAxes, ha='center')
ax.set_title(f'Generated File Bloat ({len(qw_gen_bloat)})')

# Panel 6: High codegen ratio
ax = axes[1, 2]
if not qw_high_codegen.empty:
    ax.barh(range(len(qw_high_codegen)),
            qw_high_codegen['codegen_compile_time_sum_ms'].values, color='#E8A84C')
    ax.set_yticks(range(len(qw_high_codegen)))
    ax.set_yticklabels(qw_high_codegen['cmake_target'].values, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel('Codegen compile time (ms)')
else:
    ax.text(0.5, 0.5, 'None identified', transform=ax.transAxes, ha='center')
ax.set_title(f'High Codegen Ratio (>={CODEGEN_RATIO_THRESHOLD:.0%}) ({len(qw_high_codegen)})')

plt.suptitle('Quick Wins Summary', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Summary table ────────────────────────────────────────────────────────────
print(f"Total quick wins identified: {len(quick_wins_summary)}")
print(f"Total estimated effort: {quick_wins_summary['effort_days'].sum():.1f} engineer-days\n")
print(quick_wins_summary.groupby('category').size().rename('count').to_string())
print()
display(quick_wins_summary)

## Incremental vs CI Recommendations

Separate recommendations for incremental and full-rebuild optimisation.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Incremental Build Recommendations
# ═══════════════════════════════════════════════════════════════════════════════
# Prioritise by expected daily rebuild cost — frequent changes x large blast radius.
TOP_N_INCREMENTAL = 10
top_incremental_targets = set(cost_df.head(TOP_N_INCREMENTAL)['cmake_target'])

# Candidates that affect the most costly incremental targets
incremental_recs = roi_df[
    roi_df['cmake_target'].apply(
        lambda t: any(x.strip() in top_incremental_targets for x in str(t).split(','))
    )
].copy()
incremental_recs = incremental_recs.sort_values('time_saving_ms_per_day', ascending=False)

print(f"=== Incremental Build Recommendations ===")
print(f"Focus: top {TOP_N_INCREMENTAL} targets by expected daily rebuild cost\n")
for _, r in cost_df.head(TOP_N_INCREMENTAL).iterrows():
    cp_flag = ' [CP]' if r['cmake_target'] in cp_set else ''
    print(f"  {r['cmake_target']:20s}  {r['expected_daily_cost_ms']:8.1f} ms/day  "
          f"(p={r['change_prob']:.4f}, rebuild={r['rebuild_cost_ms']:,} ms){cp_flag}")

print(f"\nActionable recommendations for incremental builds: {len(incremental_recs)}")
display(incremental_recs[['rank', 'type', 'cmake_target', 'time_saving_ms_per_day',
                           'effort_days', 'roi', 'tier']].reset_index(drop=True))

# ═══════════════════════════════════════════════════════════════════════════════
# CI / Full-Rebuild Recommendations
# ═══════════════════════════════════════════════════════════════════════════════
# Prioritise by critical path reduction — what shortens the wall-clock full build?
ci_recs = roi_df[roi_df['on_cp']].copy()
ci_recs['cp_weight_ms'] = ci_recs['cmake_target'].apply(
    lambda t: G.nodes[t].get(weight_attr, 0) if t in G else 0
)
ci_recs = ci_recs.sort_values(['roi', 'cp_weight_ms'], ascending=[False, False])

# Estimate CP reduction from top split candidates on CP
cp_impact_rows = []
cumulative_saving = 0
for _, r in ci_recs[ci_recs['type'] == 'split'].iterrows():
    t = r['cmake_target']
    if t not in G:
        continue
    saving = int(r['total_saving_ms'])
    cumulative_saving += saving
    cp_impact_rows.append({
        'cmake_target': t,
        'current_weight_ms': G.nodes[t].get(weight_attr, 0),
        'compile_saving_ms': saving,
        'projected_cp_ms': cp_length - cumulative_saving,
        'cp_reduction_pct': cumulative_saving / cp_length * 100,
    })

cp_impact_df = pd.DataFrame(cp_impact_rows) if cp_impact_rows else pd.DataFrame()

print(f"\n{'=' * 60}")
print(f"=== CI / Full-Rebuild Recommendations ===")
print(f"Current critical path: {cp_length:,} ms  ({' → '.join(cp_nodes)})")

if not cp_impact_df.empty:
    best = cp_impact_df.iloc[-1]
    print(f"Projected CP after all splits: {int(best['projected_cp_ms']):,} ms "
          f"(-{best['cp_reduction_pct']:.1f}%)")
    print()
    display(cp_impact_df)

print(f"\nAll CP recommendations ({len(ci_recs)}):")
display(ci_recs[['rank', 'type', 'cmake_target', 'time_saving_ms_per_day',
                  'total_saving_ms', 'effort_days', 'roi', 'tier']].reset_index(drop=True))

# ═══════════════════════════════════════════════════════════════════════════════
# Overlap Analysis
# ═══════════════════════════════════════════════════════════════════════════════
incr_targets = set()
for t in incremental_recs['cmake_target']:
    incr_targets.update(x.strip() for x in str(t).split(','))

ci_targets = set()
for t in ci_recs['cmake_target']:
    ci_targets.update(x.strip() for x in str(t).split(','))

both_targets = incr_targets & ci_targets
print(f"\n{'=' * 60}")
print(f"Targets appearing in BOTH incremental and CI recommendations:")
print(f"  {', '.join(sorted(both_targets)) if both_targets else 'None'}")
if both_targets:
    print("  These are the highest-priority targets overall.")

# ── Two-panel plot ───────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Panel 1: Top incremental targets by daily cost
top_incr = cost_df.head(TOP_N_INCREMENTAL).copy()
colors_incr = ['crimson' if t in cp_set else 'steelblue' for t in top_incr['cmake_target']]
ax1.barh(range(len(top_incr)), top_incr['expected_daily_cost_ms'].values, color=colors_incr)
ax1.set_yticks(range(len(top_incr)))
ax1.set_yticklabels(top_incr['cmake_target'].values, fontsize=9)
ax1.invert_yaxis()
ax1.set_xlabel('Expected daily rebuild cost (ms)')
ax1.set_title('Incremental: Top Targets by Daily Rebuild Cost')
ax1.legend(handles=[Patch(color='crimson', label='On CP'),
                    Patch(color='steelblue', label='Off CP')], loc='lower right')

# Panel 2: CP node weights — current vs projected after splits
ax2_data = pd.DataFrame({
    'target': cp_nodes,
    'current_ms': [G.nodes[n].get(weight_attr, 0) for n in cp_nodes],
})

# Apply split savings to CP nodes
split_savings_map = {}
if not cp_impact_df.empty:
    split_savings_map = cp_impact_df.set_index('cmake_target')['compile_saving_ms'].to_dict()

ax2_data['saving_ms'] = ax2_data['target'].map(split_savings_map).fillna(0).astype(int)
ax2_data['projected_ms'] = ax2_data['current_ms'] - ax2_data['saving_ms']

y = range(len(ax2_data))
ax2.barh(y, ax2_data['current_ms'], color='steelblue', alpha=0.4, label='Current')
ax2.barh(y, ax2_data['projected_ms'], color='crimson', alpha=0.8, label='After splits')
ax2.set_yticks(list(y))
ax2.set_yticklabels(ax2_data['target'].values, fontsize=9)
ax2.invert_yaxis()
ax2.set_xlabel('Effective weight (ms)')
ax2.set_title(f'CI: Critical Path — Current vs Projected')
ax2.legend(loc='lower right')

# Annotate savings
for i, (_, r) in enumerate(ax2_data.iterrows()):
    if r['saving_ms'] > 0:
        ax2.annotate(f"-{r['saving_ms']:,} ms",
                     (r['current_ms'], i), fontsize=8, ha='left', va='center',
                     color='crimson', fontweight='bold')

# CP length annotations
projected_cp = cp_length - sum(split_savings_map.values())
ax2.text(0.98, 0.02, f"Current CP: {cp_length:,} ms\nProjected: {projected_cp:,} ms "
         f"(-{(cp_length - projected_cp) / cp_length:.1%})",
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=10,
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

## Export Results

Write recommendations.csv and formatted report.


In [ ]:
from datetime import date

results_dir = Path('../data/results')
results_dir.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# Export recommendations.csv
# ═══════════════════════════════════════════════════════════════════════════════
export_cols = [
    'rank', 'type', 'cmake_target', 'detail',
    'time_saving_ms_per_day', 'total_saving_ms',
    'effort_days', 'roi', 'confidence', 'tier',
    'on_cp', 'priority',
]
export_df = roi_df[export_cols].copy()
export_df.to_csv(results_dir / 'recommendations.csv', index=False)
print(f"Saved {len(export_df)} recommendations to data/results/recommendations.csv")

# ═══════════════════════════════════════════════════════════════════════════════
# Export quick_wins.csv
# ═══════════════════════════════════════════════════════════════════════════════
quick_wins_summary.to_csv(results_dir / 'quick_wins.csv', index=False)
print(f"Saved {len(quick_wins_summary)} quick wins to data/results/quick_wins.csv")

# ═══════════════════════════════════════════════════════════════════════════════
# Formatted text report
# ═══════════════════════════════════════════════════════════════════════════════
W = 70

print(f"\n{'=' * W}")
print(f"  BUILD OPTIMISATION RECOMMENDATIONS REPORT")
print(f"  Generated: {date.today().isoformat()}")
print(f"{'=' * W}")

print(f"\nEXECUTIVE SUMMARY")
print(f"  Targets analysed:   {len(target_df)}")
print(f"  Dependency edges:   {len(edge_df)}")
print(f"  Critical path:      {cp_length:,} ms  ({' -> '.join(cp_nodes)})")
print(f"  Expected daily rebuild cost (current): {total_expected:,.0f} ms "
      f"({total_expected / 1000:.1f} s)")
print(f"  Total candidates:   {len(roi_df)}")
print(f"  Quick wins:         {len(quick_wins_summary)}")

positive_recs = roi_df[roi_df['time_saving_ms_per_day'] > 0]
if not positive_recs.empty:
    total_daily = positive_recs['time_saving_ms_per_day'].sum()
    total_effort = positive_recs['effort_days'].sum()
    print(f"  Potential daily saving: {total_daily:,.1f} ms/day ({total_daily / 1000:.2f} s/day)")
    print(f"  Total implementation effort: {total_effort:.1f} engineer-days")

# Top 5 by ROI
print(f"\n{'─' * W}")
print(f"TOP 5 RECOMMENDATIONS (by ROI)")
print(f"{'─' * W}")
for _, r in roi_df.head(5).iterrows():
    cp_flag = ' [CP]' if r['on_cp'] else ''
    print(f"\n  Rank {r['rank']} — [{r['type']}] {r['cmake_target']}{cp_flag}")
    print(f"    Daily saving: {r['time_saving_ms_per_day']:.1f} ms/day | "
          f"Effort: {r['effort_days']:.1f}d | ROI: {r['roi']:.2f} | "
          f"Confidence: {r['confidence']}")
    print(f"    {r['detail']}")

# Quick wins summary
print(f"\n{'─' * W}")
print(f"QUICK WINS")
print(f"{'─' * W}")
for cat, group in quick_wins_summary.groupby('category'):
    print(f"\n  {cat} ({len(group)} items, ~{group['effort_days'].sum():.1f}d effort):")
    for _, qw in group.iterrows():
        print(f"    - {qw['target']}: {qw['description']}")

# Incremental vs CI
print(f"\n{'─' * W}")
print(f"INCREMENTAL vs CI")
print(f"{'─' * W}")
print(f"\n  Incremental (developer rebuilds):")
print(f"    Top target: {cost_df.iloc[0]['cmake_target']} "
      f"({cost_df.iloc[0]['expected_daily_cost_ms']:.0f} ms/day)")
print(f"    Actionable recommendations: {len(incremental_recs)}")

print(f"\n  CI (full rebuilds):")
print(f"    Current CP: {cp_length:,} ms")
if not cp_impact_df.empty:
    best = cp_impact_df.iloc[-1]
    print(f"    Projected CP after splits: {int(best['projected_cp_ms']):,} ms "
          f"(-{best['cp_reduction_pct']:.1f}%)")
print(f"    CP recommendations: {len(ci_recs)}")

if both_targets:
    print(f"\n  Highest-priority (both): {', '.join(sorted(both_targets))}")

print(f"\n{'=' * W}")
print(f"  REPORT COMPLETE")
print(f"  Full details:  data/results/recommendations.csv")
print(f"  Quick wins:    data/results/quick_wins.csv")
print(f"{'=' * W}")

# ═══════════════════════════════════════════════════════════════════════════════
# Summary visualisation (2x2)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Build Optimisation Recommendations', fontsize=16, y=1.01)

# Panel 1: ROI — top 10
ax = axes[0, 0]
top10_roi = roi_df[roi_df['roi'] > 0].head(10)
if not top10_roi.empty:
    colors_roi = [TIER_COLORS[t] for t in top10_roi['tier']]
    ax.barh(range(len(top10_roi)), top10_roi['roi'].values, color=colors_roi)
    ax.set_yticks(range(len(top10_roi)))
    ax.set_yticklabels(
        [f"#{r['rank']} {r['cmake_target'][:25]}" for _, r in top10_roi.iterrows()],
        fontsize=8,
    )
    ax.invert_yaxis()
    ax.set_xlabel('ROI (ms/day per effort day)')
    ax.set_title('Top 10 by ROI')
    ax.legend(handles=[Patch(color=c, label=t) for t, c in TIER_COLORS.items()],
              loc='lower right', fontsize=7)

# Panel 2: Incremental vs CI bubble chart
ax = axes[0, 1]
plot_cost = cost_df[cost_df['expected_daily_cost_ms'] > 0].copy()
if not plot_cost.empty:
    on_cp_mask = plot_cost['cmake_target'].isin(cp_set)
    in_both = plot_cost['cmake_target'].isin(both_targets)
    colors_bubble = np.where(in_both, 'crimson', np.where(on_cp_mask, 'darkorange', 'steelblue'))
    ax.scatter(plot_cost['change_prob'], plot_cost['rebuild_cost_ms'],
               s=plot_cost['expected_daily_cost_ms'] / plot_cost['expected_daily_cost_ms'].max() * 400 + 30,
               c=colors_bubble, alpha=0.7, edgecolors='black', linewidths=0.5)
    for _, r in plot_cost.head(8).iterrows():
        ax.annotate(r['cmake_target'], (r['change_prob'], r['rebuild_cost_ms']),
                    fontsize=7, ha='left', va='bottom')
    ax.set_xlabel('Change probability (incremental)')
    ax.set_ylabel('Rebuild cost ms (CI)')
    ax.set_title('Incremental vs CI Priority')
    ax.legend(handles=[Patch(color='crimson', label='Both'),
                       Patch(color='darkorange', label='CP only'),
                       Patch(color='steelblue', label='Other')],
              loc='upper right', fontsize=7)

# Panel 3: Quick wins by category
ax = axes[1, 0]
if not quick_wins_summary.empty:
    qw_counts = quick_wins_summary.groupby('category').size().sort_values(ascending=True)
    qw_efforts = quick_wins_summary.groupby('category')['effort_days'].sum()
    effort_colors = ['#55A868' if e <= 1 else '#E8A84C' if e <= 3 else '#C44E52'
                     for e in qw_efforts.loc[qw_counts.index]]
    ax.barh(range(len(qw_counts)), qw_counts.values, color=effort_colors)
    ax.set_yticks(range(len(qw_counts)))
    ax.set_yticklabels(qw_counts.index, fontsize=8)
    ax.set_xlabel('Count')
    ax.set_title('Quick Wins by Category')
    ax.legend(handles=[Patch(color='#55A868', label='<=1d effort'),
                       Patch(color='#E8A84C', label='<=3d effort'),
                       Patch(color='#C44E52', label='>3d effort')],
              loc='lower right', fontsize=7)

# Panel 4: CP reduction projection
ax = axes[1, 1]
if not cp_impact_df.empty:
    labels = ['Current'] + cp_impact_df['cmake_target'].tolist()
    values = [cp_length] + cp_impact_df['projected_cp_ms'].tolist()
    colors_cp = ['steelblue'] + ['crimson'] * len(cp_impact_df)
    ax.bar(range(len(labels)), values, color=colors_cp, alpha=0.8)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels([f"+ split\n{l}" if i > 0 else l for i, l in enumerate(labels)],
                       fontsize=8)
    ax.set_ylabel('Critical path length (ms)')
    ax.set_title('Projected CP Reduction from Splits')
    for i, v in enumerate(values):
        ax.text(i, v + 50, f"{v:,}", ha='center', fontsize=8)
else:
    ax.text(0.5, 0.5, 'No CP split candidates', transform=ax.transAxes, ha='center')
    ax.set_title('Projected CP Reduction')

plt.tight_layout()
plt.show()

display(export_df.head(10))